In [31]:
import re
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

df = pd.read_csv("../../data/processed/combined_videos_raw.csv", low_memory=False)

df["video_publishedAt"] = pd.to_datetime(df["video_publishedAt"], utc=True, errors="coerce")
df["snapshot_utc"]      = pd.to_datetime(df["snapshot_utc"],      utc=True, errors="coerce")

def parse_duration(val):
    if pd.isna(val):
        return np.nan
    m = re.fullmatch(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", str(val).strip())
    if not m:
        return np.nan
    h, mi, s = (int(x) if x else 0 for x in m.groups())
    return float(h * 3600 + mi * 60 + s)

df["duration_sec"] = df["duration"].apply(parse_duration)
df["is_short"]     = df["duration_sec"] <= 60

for col in ["views", "likes", "comments"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded: 28,037 rows × 56 columns


In [32]:
drop_log = {}

mask_no_views = df["views"].isna()
drop_log["views null"] = mask_no_views.sum()

mask_bad_dur = df["duration_sec"].isna()
drop_log["duration unparseable"] = mask_bad_dur.sum()

df.loc[df["video_description"].str.strip().eq(""), "video_description"] = np.nan

drop_mask = mask_no_views | mask_bad_dur
df = df[~drop_mask].reset_index(drop=True)

for reason, n in drop_log.items():
    print(f"  {reason:30s}: {n}")
print(f"  {'Total dropped':30s}: {drop_mask.sum()}")
print(f"\nShape after drops: {df.shape}")

  views null                    : 1
  duration unparseable          : 36
  Total dropped                 : 37

Shape after drops: (28000, 56)


In [33]:
df["like_rate"]     = df["likes"]    / df["views"].replace(0, np.nan)
df["comment_rate"]  = df["comments"] / df["views"].replace(0, np.nan)
df["views_per_day"] = df["views"]    / (
    (df["snapshot_utc"] - df["video_publishedAt"]).dt.total_seconds() / 86_400
).replace(0, np.nan)

df["publish_year"]  = df["video_publishedAt"].dt.year
df["publish_month"] = df["video_publishedAt"].dt.to_period("M").dt.to_timestamp()

In [ ]:
_BOILERPLATE_RE = re.compile(
    r"""
    \bsubscribe\b | \bhit\s+the\s+bell\b | \bturn\s+on\s+notifications?\b
    | \bsmash\s+(that\s+)?like\b | \blike\s+and\s+subscribe\b | \blike\s+(and|&)\s+share\b
    | \bbecome\s+a\s+channel\s+member\b
    | \bpatreon\b | \bko-?fi\b | \bbuy\s+me\s+a\s+coffee\b | \bbecome\s+a\s+patron\b
    | \bsupport\s+(the|this|my)\s+(channel|show|podcast)\b | \bmerch\b | \bmerchandise\b
    | \bjoin\s+(my|our|the)\s+discord\b | \bdiscord\s+(server|community|link)\b | \bdiscord\.gg\b
    | \bsponsored\s+by\b | \bthis\s+(episode|video)\s+(was\s+|is\s+)?sponsored\b
    | \bbrought\s+to\s+you\s+by\b | \buse\s+(code|promo\s+code|discount\s+code)\b
    | \baffiliate\s+(link|code|partner)\b | \bpromo\s+code\b | \bdiscount\s+code\b
    | \bfollow\s+(me|us)\s+on\b | \bfind\s+(me|us)\s+on\b | \bcheck\s+(me|us)\s+out\s+on\b
    | \bmy\s+(instagram|twitter|tiktok|facebook|twitch)\s*[:@]
    | \b(instagram|twitter|tiktok|facebook)\s*:\s*[@]?\w+
    | \blinks?\s+(below|in\s+(the\s+)?description|in\s+bio)\b
    | \bmore\s+info\s+(below|in\s+the\s+description)\b | \ball\s+links?\s+in\b
    | \bbusiness\s+(inquir|email|contact)\b
    | \bfor\s+(sponsorship|collaboration|collab|business)\s+(inquir|email|contact|opportunit)
    | \b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b
    | \ball\s+rights\s+reserved\b | \bcopyright\s+\u00a9?\s*\d{4}\b
    | \bthank\s*(s|\s+you)\s+for\s+watching\b | \bsee\s+you\s+(in\s+the\s+next|next\s+time)\b
    | ^(chapters?|timestamps?)\s*:?\s*$
    | \btwitch\.tv\b | \blive\s+stream\b | \bgoing\s+live\b | \bwatch\s+live\b
    | \bsubscribe\s+for\s+more\b | \bnotification\s+squad\b
    | \bdonat(e|ions?)\b | \btip\s+(jar|link)\b | \bstreamlabs\b
    | \btts\b | \bchannel\s+points?\b | \bsubs?\s+only\b
    | \bsmarter\s+every\s+day\b | \bsmartereveryday\b
    | \bday\s+(instagram|facebook|twitter|snapchat)\b
    | \b(instagram|facebook|twitter|snapchat)\s+smarter\b
    | \bnord\s*vpn\b | \bexpressvpn\b | \bsurfshark\b | \bvpn\s+(deal|sponsor|code|link)\b
    | \buse\s+code\b | \bget\s+\d+%\s+off\b | \bfirst\s+\d+\s+people\b
    | \bnebula\b | \bepidemic\s+sound\b | \bartlist\b
    | \bsoundcloud\b | \bbandcamp\b | \bspotify\b
    | \bstream\s+(on|it|now)\b | \bavailable\s+on\b
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)

def strip_boilerplate(text) -> str:
    if pd.isna(text) or not str(text).strip():
        return ""
    lines = str(text).splitlines()
    kept = [ln.strip() for ln in lines if ln.strip() and not _BOILERPLATE_RE.search(ln)]
    return " ".join(kept)

URL_RE     = re.compile(r"http\S+|www\.\S+")
HASHTAG_RE = re.compile(r"#\w+")
MENTION_RE = re.compile(r"@\w+")
EMOJI_RE   = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F9FF"
    "\U00002700-\U000027BF"
    "\U0001FA00-\U0001FA6F"
    "]+",
    flags=re.UNICODE,
)
PUNCT_RE = re.compile(r"[^\w\s]")
SPACE_RE = re.compile(r"\s+")

def clean_text(text: str | None) -> str:
    if pd.isna(text) or not str(text).strip():
        return ""
    text = URL_RE.sub(" ", text)
    text = HASHTAG_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = EMOJI_RE.sub(" ", text)
    text = PUNCT_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text)
    return text.strip()

df["title_clean"] = df["video_title"].apply(clean_text)
df["desc_clean"]  = df["video_description"].apply(strip_boilerplate).apply(clean_text)
df["tags_clean"]  = (
    df["video_tags"]
    .fillna("")
    .apply(lambda x: " ".join(clean_text(tag) for tag in x.split("|") if tag.strip()))
)

In [35]:
df["bertopic_text"] = (
    df["title_clean"] + " "
    + df["title_clean"] + " "
    + df["desc_clean"]  + " "
    + df["tags_clean"]
).str.strip()

df["bertopic_text"] = df["bertopic_text"].str.replace(r"\b\d+\b", " ", regex=True)
df["bertopic_text"] = df["bertopic_text"].str.replace(r"\s+", " ", regex=True).str.strip()
df["bertopic_token_count"] = df["bertopic_text"].str.split().str.len()

SPARSE_THRESHOLD     = 5
df["is_sparse_text"] = df["bertopic_token_count"] < SPARSE_THRESHOLD

In [36]:
DROP_COLS = [
    "thumb_default_url", "thumb_default_width", "thumb_default_height",
    "thumb_medium_url",  "thumb_medium_width",  "thumb_medium_height",
    "thumb_high_url",    "thumb_high_width",     "thumb_high_height",
    "thumb_standard_url","thumb_standard_width", "thumb_standard_height",
    "thumb_maxres_url",  "thumb_maxres_width",   "thumb_maxres_height",
    "video_description", "video_tags",
    "video_channelId", "uploads_playlist_id", "channel_topicIds", "video_topicIds",
    "channel_isLinked", "channel_madeForKids", "projection", "embeddable",
]

df_clean = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
print(f"Columns: {df.shape[1]} -> {df_clean.shape[1]}")

Columns: 67 -> 42


In [37]:
df_clean.to_csv("../../data/processed/combined_videos_clean.csv", index=False)
df_clean["bertopic_text"].to_csv("../../data/processed/bertopic_corpus.txt", index=False, header=False)

meta_cols = [
    "video_id", "video_title", "channel_title", "category_name",
    "publish_year", "views", "like_rate", "comment_rate",
    "duration_sec", "is_short", "is_sparse_text", "bertopic_token_count",
]
df_clean[meta_cols].to_csv("../../data/processed/bertopic_metadata.csv", index=False)

In [38]:
has_desc = df_clean["desc_clean"].ne("")
has_tags = df_clean["tags_clean"].ne("")

print(f"Rows: {len(df_clean):,}")
print(f"Title + desc + tags: {(has_desc & has_tags).sum():,} ({(has_desc & has_tags).mean()*100:.1f}%)")
print(f"Title + desc only: {(has_desc & ~has_tags).sum():,} ({(has_desc & ~has_tags).mean()*100:.1f}%)")
print(f"Title + tags only: {(~has_desc & has_tags).sum():,} ({(~has_desc & has_tags).mean()*100:.1f}%)")
print(f"Title only: {(~has_desc & ~has_tags).sum():,} ({(~has_desc & ~has_tags).mean()*100:.1f}%)")
print(f"Sparse (< 5 tokens): {df_clean['is_sparse_text'].sum():,} ({df_clean['is_sparse_text'].mean()*100:.1f}%)")

Rows: 28,000
Title + desc + tags: 19,231 (68.7%)
Title + desc only: 5,298 (18.9%)
Title + tags only: 647 (2.3%)
Title only: 2,824 (10.1%)
Sparse (< 5 tokens): 267 (1.0%)


In [39]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from pathlib import Path

RANDOM_STATE   = 42
MIN_TOPIC_SIZE = 30
N_GRAM_RANGE   = (1, 2)
TOP_N_WORDS    = 10
MODEL_DIR      = Path("../../outputs/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [40]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

_CHANNEL_STOPWORDS = [
    "smarter", "smartereveryday", "vsauce", "veritasium", "kurzgesagt",
    "mrbeast", "markiplier", "pewdiepie", "linus", "linustechtips",
    "technicalguruji", "mkbhd", "unboxtherapy", "jerryrigeverything",
    "jacksepticeye", "affiliate",
]

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_STATE,
    low_memory=False,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=MIN_TOPIC_SIZE,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

vectorizer_model = CountVectorizer(
    ngram_range=N_GRAM_RANGE,
    stop_words=list(ENGLISH_STOP_WORDS) + _CHANNEL_STOPWORDS,
    min_df=10,
    max_df=0.85,
)

representation_model = KeyBERTInspired()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [41]:
topic_model = BERTopic(
    embedding_model = embedding_model,
    umap_model = umap_model,
    hdbscan_model = hdbscan_model,
    vectorizer_model = vectorizer_model,
    representation_model = representation_model,
    top_n_words = TOP_N_WORDS,
    nr_topics = 90,
    calculate_probabilities = False,
    verbose = True,
)

corpus = df_clean["bertopic_text"].tolist()
topics, _ = topic_model.fit_transform(corpus)

print(f"Topics found (excl. -1) : {len(set(topics)) - 1}")
print(f"Outlier docs (topic -1) : {topics.count(-1):,} ({topics.count(-1)/len(topics)*100:.1f}%)")

2026-03-11 12:05:59,963 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/875 [00:00<?, ?it/s]

2026-03-11 12:08:41,843 - BERTopic - Embedding - Completed ✓
2026-03-11 12:08:41,843 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-11 12:08:55,760 - BERTopic - Dimensionality - Completed ✓
2026-03-11 12:08:55,761 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-11 12:08:57,458 - BERTopic - Cluster - Completed ✓
2026-03-11 12:08:57,459 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-03-11 12:08:59,593 - BERTopic - Representation - Completed ✓
2026-03-11 12:08:59,594 - BERTopic - Topic reduction - Reducing number of topics
2026-03-11 12:08:59,606 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-11 12:09:08,577 - BERTopic - Representation - Completed ✓
2026-03-11 12:09:08,580 - BERTopic - Topic reduction - Reduced number of topics from 227 to 90


Topics found (excl. -1) : 89
Outlier docs (topic -1) : 11,005 (39.3%)


In [42]:
new_topics = topic_model.reduce_outliers(corpus, topics, strategy="embeddings", threshold=0.3)
topic_model.update_topics(corpus, topics=new_topics, vectorizer_model=vectorizer_model, representation_model=representation_model)

outlier_before = topics.count(-1)
outlier_after  = list(new_topics).count(-1)
print(f"Outliers before: {outlier_before:,} ({outlier_before/len(topics)*100:.1f}%)")
print(f"Outliers after: {outlier_after:,} ({outlier_after/len(topics)*100:.1f}%)")
print(f"Reclaimed: {outlier_before - outlier_after:,}")

topics = new_topics

2026-03-11 12:10:11,996 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outliers before: 11,005 (39.3%)
Outliers after: 965 (3.4%)
Reclaimed: 10,040


In [43]:
df_clean["topic_id"] = topics

topic_info   = topic_model.get_topic_info()
topic_labels = (
    topic_info[topic_info["Topic"] != -1][["Topic", "Name"]]
    .rename(columns={"Topic": "topic_id", "Name": "topic_label"})
)

def dominant_share(d):
    vals = list(d.values())
    return max(vals) / sum(vals)

topic_stats = (
    df_clean[df_clean["topic_id"] != -1]
    .groupby("topic_id")
    .agg(
        video_count = ("video_id", "count"),
        median_views = ("views", "median"),
        median_like_rate = ("like_rate", "median"),
        top_category = ("category_name", lambda x: x.value_counts().index[0]),
        category_mix = ("category_name", lambda x: x.value_counts().to_dict()),
    )
    .reset_index()
    .merge(topic_labels, on="topic_id")
)

topic_stats["dominant_category_share"] = topic_stats["category_mix"].apply(dominant_share)
topic_stats = topic_stats.sort_values("median_views", ascending=False).reset_index(drop=True)

print(f"Topics (excl. -1): {len(topic_labels)}")
print(topic_stats.head(15)[["topic_id", "topic_label", "video_count", "median_views", "top_category"]].to_string(index=False))

Topics (excl. -1): 89
 topic_id                                             topic_label  video_count  median_views     top_category
       64                        64_order_food_black friday_games           67     7258596.0    entertainment
       16       16_nile blue_chemical_science chemistry_chemistry          429     4750183.0 research_science
       15                 15_youtuber_youtubers_newsletter_twitch          438     2307420.5           gaming
       71                       71_roger_restaurant_chef_brothers           49     1905103.0             food
       81          81_season episode_episode_episode check_finale           45     1698303.0           gaming
       29 29_practical engineering_engineering_engineers_engineer          261     1616256.0 research_science
       32      32_scientists_researchers_scientist_science future          260     1267396.5 research_science
       20       20_nebula_nebula using_free nebula_epidemic sound          373      892861.0 resea

In [44]:
outlier_rate = (df_clean["topic_id"] == -1).mean()
print(f"Outlier rate: {outlier_rate*100:.1f}%  (target < 20%)")
print(f"Topics < 50: {(topic_stats['video_count'] < 50).sum()}")
print(f"Median dom share: {topic_stats['dominant_category_share'].median():.2f}")
print(f"Topics > 80% 1 cat: {(topic_stats['dominant_category_share'] > 0.8).sum()}")
print(f"Cross-category: {(topic_stats['dominant_category_share'] <= 0.5).sum()}")

for tid in topic_stats.sample(3, random_state=RANDOM_STATE)["topic_id"].tolist():
    kw_str = ", ".join([w for w, _ in topic_model.get_topic(tid)[:8]])
    examples = df_clean[df_clean["topic_id"] == tid]["video_title"].sample(
        min(3, (df_clean["topic_id"] == tid).sum()), random_state=RANDOM_STATE
    ).tolist()
    print(f"\n  Topic {tid} | {kw_str}")
    for ex in examples:
        print(f" - {ex}")

Outlier rate: 3.4%  (target < 20%)
Topics < 50: 4
Median dom share: 0.87
Topics > 80% 1 cat: 59
Cross-category: 6

  Topic 23 | security, privacy, surveillance, hackers, hacking, cyber, vpn, internet
 - how to hack a telescope | ransomware sucks
 - Don’t Blame Signal - The Real Story Behind the TM SGNL Breach
 - this isn't great...

  Topic 34 | copyright, section copyright, video essay, media, rights, law, manipulation, influencers
 - Doterra Influencer Scams Grieving Mothers & More WILD MLM Stories…
 - the MOST manipulative family channel... (Della Vlogs)
 - Audio Episode 02 - The Case of The Cursed Jungle Cruise

  Topic 58 | cod, battlefield, gameplay, doom, zombies, game review, beta, warfare
 - #Battlefield 1 Revolution Hidden in Plain Sight #Part One #TheBrokenMachineGun
 - ASSASSIN'S CREED FREEDOM CRY Gameplay Walkthrough FULL GAME [4K 60FPS PC ULTRA] - No Commentary
 - Oblivion Has Aged Like Wine #oblivionremaster


In [45]:
topic_model.save(str(MODEL_DIR / "bertopic_model"), serialization="pickle", save_ctfidf=True)

topic_stats.to_csv("../../data/processed/topic_stats.csv", index=False)

meta_out = (
    df_clean[[
        "video_id", "video_title", "channel_title", "category_name",
        "publish_year", "views", "like_rate", "comment_rate",
        "duration_sec", "is_short", "is_sparse_text", "bertopic_token_count", "topic_id",
    ]]
    .copy()
    .merge(topic_labels, on="topic_id", how="left")
)
meta_out["topic_label"] = meta_out["topic_label"].fillna("outlier")
meta_out.to_csv("../../data/processed/bertopic_metadata.csv", index=False)

print(f"Topics found: {len(set(topics)) - 1}")
print(f"Outlier docs (topic -1): {list(topics).count(-1):,} ({list(topics).count(-1)/len(topics)*100:.1f}%)")
print(f"Median topic size: {topic_stats['video_count'].median():.0f} videos")
print(f"Largest topic: {topic_stats['video_count'].max()} videos")

2026-03-11 12:10:20,813 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Topics found: 89
Outlier docs (topic -1): 965 (3.4%)
Median topic size: 171 videos
Largest topic: 3270 videos
